In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-rdd-preparation").getOrCreate()
sc = spark.sparkContext
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 13:39:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Spark RDD

Readings

- https://spark.apache.org/docs/latest/rdd-programming-guide.html
- https://spark.apache.org/examples.html RDD subsection

## Overview

A **Resilient Distributed Dataset (RDD)** is Spark's original low-level distributed data structure.

Key properties:
- **Resilient**: Fault-tolerant -- if a partition is lost, Spark recomputes it from the *lineage* (the recorded chain of transformations that produced it)
- **Distributed**: Automatically partitioned across cluster nodes
- **Dataset**: Can hold any serializable Python object

RDDs use **lazy evaluation**: `map()`, `filter()`, `flatMap()` etc. only *record* the transformation -- nothing executes until an **action** (`collect()`, `count()`, `saveAsTextFile()`, …) is triggered. This lets Spark optimize the full pipeline as a DAG before running.

**When to use RDDs vs DataFrames:**
- Use **DataFrames** for structured/tabular data -- the Catalyst optimizer makes them much faster
- Use **RDDs** for unstructured data, custom Python objects, or fine-grained partition control

# Example 1: Calculating Pi

https://github.com/apache/spark/blob/master/examples/src/main/python/pi.py

## Concept: Monte Carlo Simulation with RDD

The Monte Carlo method estimates π by random sampling:
- A random point (x, y) in the unit square is *inside* the inscribed circle if x² + y² ≤ 1
- The probability of falling inside is π/4, so: **π ≈ 4 × (inside / total)**

`sc.parallelize(range(n), partitions)` distributes the work across multiple tasks.  
Each task runs `f(i)` independently -- no communication during the map phase.  
Only the final `reduce(add)` needs coordination (a single sum across partitions).

**In a cluster**, these partitions would run on different machines simultaneously.

In [11]:
from random import random
from functools import reduce
from operator import add

In [3]:
add(1,2)

3

In [4]:
partitions = 10
n = 100000 * partitions

In [5]:
def f(_: int) -> float:
    x = random() * 2 - 1
    y = random() * 2 - 1
    return 1 if x ** 2 + y ** 2 <= 1 else 0

In [6]:
f(1)

1

In [12]:
# using Python
data = range(1, n+1)
inside = map(f, data)
count = reduce(add, inside)
count

784828

In [13]:
print("Pi is roughly %f" % (4.0 * count / n))

Pi is roughly 3.139312


In [7]:
# using Spark
rdd = spark.sparkContext.parallelize(range(1, n + 1), partitions)
rdd

PythonRDD[1] at RDD at PythonRDD.scala:58

In [8]:
count = rdd.map(f).reduce(add)
count

785100

In [9]:
print("Pi is roughly %f" % (4.0 * count / n))

Pi is roughly 3.140400


# Example 2: Word Count

https://github.com/apache/spark/blob/master/examples/src/main/python/wordcount.py

## Concept: MapReduce Pattern

Word Count is the "Hello World" of distributed computing. It demonstrates the **MapReduce** pattern:

1. `sc.textFile(...)` -- reads the file as an RDD of lines
2. `flatMap(lambda x: x.split(' '))` -- splits each line into words (one → many)
3. `map(lambda x: (x, 1))` -- creates `(word, 1)` key-value pairs
4. `reduceByKey(add)` -- sums all 1s for the same key -- **this triggers a shuffle** (data moves between partitions)
5. `collect()` / `saveAsTextFile()` -- actions that trigger execution of the whole pipeline

`toDebugString()` reveals Spark's **DAG**: notice the `ShuffledRDD` boundary -- that is the reduce phase.

In [41]:
us_const = os.path.join(os.getcwd(), "data/US-Constitution.txt")
us_const

'/app/work/data-engineering-preparation/data/US-Constitution.txt'

In [14]:
with open(us_const) as f:
    lines = f.readlines()

In [15]:
lines[:10]


['\n',
 '  The Constitution of the United States of America\n',
 '\n',
 '\n',
 'Preamble\n',
 '\n',
 'We the people of the United States, in order to form a\n',
 'more perfect union, establish justice, insure domestic\n',
 'tranquillity, provide for the common defense, promote\n',
 'the general welfare, and secure the blessings of\n']

In [16]:
lines = sc.textFile(us_const) # reading with spark, distributed

In [17]:
counts = lines.flatMap(lambda x: x.split(' ')) \
              .map(lambda x: (x, 1)) \
              .reduceByKey(add)
counts

PythonRDD[9] at RDD at PythonRDD.scala:58

In [18]:
print(counts.toDebugString().decode('utf-8')) # show the plan

(2) PythonRDD[9] at RDD at PythonRDD.scala:58 []
 |  MapPartitionsRDD[8] at mapPartitions at PythonRDD.scala:170 []
 |  ShuffledRDD[7] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[6] at reduceByKey at /tmp/ipykernel_25/729497489.py:3 []
    |  PythonRDD[5] at reduceByKey at /tmp/ipykernel_25/729497489.py:3 []
    |  data/US-Constitution.txt MapPartitionsRDD[4] at textFile at NativeMethodAccessorImpl.java:0 []
    |  data/US-Constitution.txt HadoopRDD[3] at textFile at NativeMethodAccessorImpl.java:0 []


In [19]:
output = counts.collect()

In [20]:
output[:5]

[('', 412), ('of', 297), ('United', 55), ('States', 20), ('America', 2)]

# Example 3: Tweet Analysis

## Dataset: Joe Biden 2020 Tweets

The file `data/joe_biden_tweets_2020.csv` contains tweets from Biden's campaign account in 2020.  
Columns: `timestamp, url, tweet, replies, retweets, quotes, likes`

Because the tweet text contains commas inside quotes, we use `re` (regular expressions) to extract fields reliably instead of splitting on commas.

This example demonstrates:
- Filtering an RDD with a predicate (`filter`)
- Extracting structured data from free-form text with `re.compile(pattern).findall()`
- Aggregating with `countByValue()` -- returns a Python `defaultdict` (an action)

In [21]:
tweets = sc.textFile(os.path.join(os.getcwd(), "data/joe_biden_tweets_2020.csv"))

In [22]:
tweets.count()

5759

In [23]:
first5 = tweets.take(5)
type(first5)

list

In [24]:
first5 # notice first one is header

['timestamp,url,tweet,replies,retweets,quotes,likes',
 '2020-01-01 01:15,https://twitter.com/JoeBiden/status/1212180387260010496,"Our final fundraising deadline of 2019 is just hours away and we need your help. Every donation — big or small — goes a long way to helping us make Donald Trump a one-term president.',
 '',
 'Chip in before midnight to help us reach our goal: https://t.co/CznFJFHT2T",411,269,32,948',
 '2020-01-01 18:35,https://twitter.com/JoeBiden/status/1212442112219844609,"Every single human being deserves to be treated with dignity. Everyone. The poor and the powerless, the marginalized and vulnerable, the “least of these.” That has been the animating principle of my life and my faith. https://t.co/BwmOVQjxVk",1136,2423,182,11574']

In [25]:
# how many tweets contain @realDonaldTrump (in any casing)
q1 = tweets.filter(lambda t: "@realdonaldtrump" in t.lower())
q1

PythonRDD[14] at RDD at PythonRDD.scala:58

In [26]:
q1.count()

15

In [27]:
# show the first five tweets
q1.take(5)

['2020-01-05 19:40,https://twitter.com/JoeBiden/status/1213908021304332288,"Hey @realDonaldTrump, do you think the American people want another war in the Middle East? Des Moines certainly doesn’t. https://t.co/uFgQuqg1oL",1308,3235,178,14803',
 '2020-01-15 23:05,https://twitter.com/JoeBiden/status/1217583673282789377,"This seems like a great reason to rejoin the Paris Agreement, @realDonaldTrump. https://t.co/HHI4KUokcF",343,1595,37,8544',
 '2020-02-05 20:20,https://twitter.com/JoeBiden/status/1225152126613118981,"Fred, it must not have been easy to sit there and listen to @realdonaldtrump lie. I’m proud of you for speaking truth to the abuse of power. https://t.co/xrnSAKC8QZ",1038,8017,338,44879',
 '2020-02-15 00:32,https://twitter.com/JoeBiden/status/1228477020012601344,"Wanted to make sure you saw this, @realDonaldTrump: “Trump’s First 3 Years Created 1.5 Million Fewer Jobs Than Obama’s Last 3.” https://t.co/KWoHe1d4ku",1521,13961,627,40497',
 '2020-03-04 04:07,https://twitter.com/

In [28]:
t = _[0]

In [29]:
import re

In [31]:
regex = re.compile(r'\d,"(.+)",\d') # match text between digit , "

In [32]:
regex.findall(t)[0]

'Hey @realDonaldTrump, do you think the American people want another war in the Middle East? Des Moines certainly doesn’t. https://t.co/uFgQuqg1oL'

In [33]:
q1.map(lambda t: regex.findall(t)[0]).take(5)


['Hey @realDonaldTrump, do you think the American people want another war in the Middle East? Des Moines certainly doesn’t. https://t.co/uFgQuqg1oL',
 'This seems like a great reason to rejoin the Paris Agreement, @realDonaldTrump. https://t.co/HHI4KUokcF',
 'Fred, it must not have been easy to sit there and listen to @realdonaldtrump lie. I’m proud of you for speaking truth to the abuse of power. https://t.co/xrnSAKC8QZ',
 'Wanted to make sure you saw this, @realDonaldTrump: “Trump’s First 3 Years Created 1.5 Million Fewer Jobs Than Obama’s Last 3.” https://t.co/KWoHe1d4ku',
 'You lost tonight, @realDonaldTrump. Democrats around the country are fired up. We are decent, brave, and resilient people. We are better than you. Come November, we are going to beat you. https://t.co/gjZcrcCDlX']

In [34]:
# how many tweets in each month?
t

'2020-01-05 19:40,https://twitter.com/JoeBiden/status/1213908021304332288,"Hey @realDonaldTrump, do you think the American people want another war in the Middle East? Des Moines certainly doesn’t. https://t.co/uFgQuqg1oL",1308,3235,178,14803'

In [35]:
regex2 = re.compile(r'^(2020-\d\d)-\d\d ') 

In [36]:
regex2.findall(t)
    

['2020-01']

In [37]:
q2 = tweets.map(lambda t: regex2.findall(t)).filter(lambda lst: len(lst) > 0).map(lambda lst: lst[0])

In [38]:
q2.take(5)

['2020-01', '2020-01', '2020-01', '2020-01', '2020-01']

In [39]:
q2.countByValue()

defaultdict(int,
            {'2020-01': 201,
             '2020-02': 277,
             '2020-03': 324,
             '2020-04': 266,
             '2020-05': 250,
             '2020-06': 219,
             '2020-07': 228,
             '2020-08': 337,
             '2020-09': 303,
             '2020-10': 504,
             '2020-11': 19})

# Shutdown spark when done

In [40]:
spark.stop()